In [ ]:
import os
from eduray import *
from eduray.internals import *
from typing import List, Tuple

# setup for notebooks
if not os.path.exists("images"):
    try:
        os.makedirs("images")
    except FileExistsError:
        pass


# Introduction to data representation
- Welcome to the first notebook of this series! In this notebook, we will be exploring how to represent data in our ray tracer. We will be defining some basic data structures that we will be using throughout the rest of the notebooks. We will also be implementing some basic operations on these data structures. Let's get started!

> You can skip this if you want to jump straight into the ray-tracing implementation, but I recommend reading through it to get a better understanding of how we will be representing our data.
### What is color and how is it represented?

In computer graphics, color is commonly represented using the RGB color model, which combines red, green, and blue components. Each channel can be stored either as an integer in the range 0 to 255 or as a floating-point value in the range 0.0 to 1.0, where 0.0 means no intensity and 1.0 means the maximum intensity of the given component. In this notebook, the floating-point representation is used because it is simple and convenient for further calculations.

Many other color models also exist, such as CMYK, HSV, or HDR formats, but for this series of notebooks the RGB model is sufficient. It is easy to understand, straightforward to implement, and widely used in computer graphics.

## What is colour, mathematically?

In these notebooks, a colour is represented as a triple of floating-point values $(r, g, b) \in [0,1]^3$, where each component gives the intensity of the red, green, and blue channel.

Floating-point values are used instead of 8-bit integers because intermediate results in rendering computations may temporarily become smaller than 0.0 or larger than 1.0. This happens, for example, when combining light intensities with material coefficients. For this reason, values are clamped to the interval $[0,1]$ only at the final stage, when the colour is written into the output image.

This representation is implemented by the Color class from src. In the current notebook, it is used mainly for storing and displaying colour values, while the actual colour computations are introduced later in the shading sections. Since the class is based on np.ndarray, it can also be extended with custom functionality if needed.

### Image as a 2D grid of colours
An image with resolution $w \times h$ can be understood as a two-dimensional array of pixels. Each pixel $(i, j)$ stores a Color value. The coordinate origin $(0, 0)$ lies in the top-left corner of the image:

- $i$ increases from left to right
- $j$ increases from top to bottom

In [ ]:
Image = Tuple[List[Color], int, int]
white = Color.custom_rgb(255, 255, 255) # defined by u8 values, but stored as linear RGB floats
black = Color.linear_rgb(0.0, 0.0, 0.0) # defined by linear RGB floats

def create_empty_image(width: int, height: int) -> Image:
    """
    Creates an empty image with the given width and height, filled with black pixels.

    :param width: The width of the image in pixels
    :param height: The height of the image in pixels
    :return: A tuple containing the list of pixel colors, width, and height
    """
    pixels = [black for _ in range(width * height)]
    return pixels, width, height

### Let's create a simple image and display it in the notebook
- this is handled internally by the ral library but this is the concept of simple image saving pipeline.
> PPM is a simple image format that stores pixels as 3 bytes (R, G, B) for each pixel. It is easy to write and read. We can also convert it to PNG format that is better supported and compressed.

In [ ]:
image = create_empty_image(500, 500)

# we first crate ppm file that represents color data of our image, then we convert it to png format and display it in the notebook
image_to_ppm("images/black_image.ppm", image)
convert_ppm_to_png("images/black_image.ppm", "images/black_image.png")
ipynb_display_images("images/black_image.png")

We can also interpret an image as a 3D arrangement of colour values, where the horizontal and vertical axes correspond to pixel positions and the third axis represents the colour assigned to each pixel. By iterating over all pixel coordinates and assigning colours based on their position, it is possible to generate a simple gradient image.

The following example creates a `50 × 50` image in which each pixel colour depends on its `x` and `y` coordinates. This produces a smooth transition across the image. The generated images are then displayed in a grid inside the notebook.

In [ ]:
def set_pixel_col(image: Image, x: int, y: int, color: Color) -> None:
    """
    Sets the color based on provided x and y coordinates in the image. The function checks if the provided coordinates are within the bounds of the image dimensions before setting the color for the specified pixel.
    :param image: The image represented as a tuple of (pixels, width, height)
    :param x: pixel's x-coordinate (horizontal position)
    :param y: pixel's y-coordinate (vertical position)
    :param color: color to set for the specified pixel, represented as a tuple of (R, G, B) values
    :return: None
    """
    pixels, width, height = image
    if 0 <= x < width and 0 <= y < height:
        pixels[y * width + x] = color

### Represent depth as a series of images

In [ ]:
images = []
image = create_empty_image(50, 50)

for z in range(50):
    for x in range(50):
        for y in range(50):
                r = x / 49.0
                g = y / 49.0
                b = z / 49.0
                set_pixel_col(image, x, y, Color.linear_rgb(r, g, b))
    images.append(image_pipeline(image, z))

ipynb_display_multiple_images_in_row(images, row_size=10)

***
## Coordinate system

This ray tracer uses a **right-handed coordinate system**:

| Axis | Direction |
|------|-----------|
| $+X$ | right |
| $+Y$ | up |
| $+Z$ | toward the viewer (out of the screen) |

The camera looks along $-Z$ by default. All scene objects, the camera, and light sources
are defined in this **world space**. The `Visualizer` below lets you explore this system
interactively — try changing `size` or rotating the view.

***
# Let us visualize the coordinate system.

You can adjust the settings of the `create_empty_scene` function to change the appearance of the scene as needed. A complete example is shown below.

This function is used later to create the initial empty scenes for demonstrating ray-tracing concepts, so it is useful to try it first and understand how it works. Recreating the scene each time is intentional, because it clears the previous visualizer output and provides a clean base for drawing the next steps of the ray-tracing process, such as the camera, rays, and intersections.

In [ ]:
vis = Visualizer()
vis.create_empty_scene(
    size=4.0,
    figsize=(12, 8),
    show_axes_labels=True,
    show_arrows=True,
    show_grid=True,
    show_axes=True,
    show_xyz_labels=True,
    background_color="whitesmoke",
)
vis.savefig("images/right_handed_coordinate_system.pdf")
vis.show()

### Different example of background grid
> *Try it out by changing the parameters of `create_empty_scene` function. You can also change the size of the grid, the spacing between the lines, and the colors used for the grid and axes. This is a good opportunity to experiment with the visualizer and get a better understanding of how it works.*

In [ ]:
vis.create_empty_scene(
    size=3.0,
    figsize=(12, 8),
    show_axes_labels=False,
    show_arrows=False,
    show_grid=False,
    show_axes=True,
    show_xyz_labels=False,
    background_color="white",
    view_elev=30,
    view_azim=-40,
    view_roll=0,
)
vis.savefig("images/right_handed_coordinate_system_2.pdf")
vis.show()

## Summary

In this notebook, we introduced the basic image definition and visual foundations needed for the later implementation of a ray tracer. We covered the following topics:

- **Colour representation** — RGB colour stored as floating-point values in the interval $[0, 1]$
- **Image representation** — image stored as a flat list of pixel colours together with width and height
- **Image output** — converting the internal image representation to displayable output formats
- **Simple image construction** — creating black images and colour gradients for testing and visualization
- **Coordinate system** — right-handed world space with $+X$ to the right, $+Y$ up, and $Z$ toward the viewer (out of the screen). Using the `Visualizer` to explore the coordinate system interactively.

**Next:**
- **Pinhole camera** — position, orthonormal basis ($\mathbf{forward}$, $\mathbf{right}$, $\mathbf{up}$), field of view, projection plane
- **Ray generation** — pixel $(i, j)$ → NDC $(u, v)$ → normalised ray direction
- **Render loop** — sequential, supersampled, and randomised pixel traversal strategies that can be further modified and experimented with